<a href="https://colab.research.google.com/github/Meenakshi-N-24/RedFlag/blob/main/notebooks/05_explanation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RedFlag - 05: Explanation layer and audit
This notebook holds the code that produced the 20-message explanation audit described in Section 8 of the report: the knowledge-base chunks, the Gemini prompt, and the resumable audit loop.

**The saved evidence is `RedFlag_Audit_Sheet.xlsx`, not a re-run.** The audit used the free Gemini tier, which allowed about 20 requests per day for the model used at the time of testing, and Gemini's wording varies between runs, so a new run would give different text and would not match the ratings in the sheet. The last section of this notebook recomputes the audit summary from the rated sheet.

Files needed in Colab to run the audit loop: `indian_test_set.csv`, `final_tfidf.joblib`, `final_lr.joblib` (made by notebook 04), the four knowledge-base `.txt` files, and a Colab secret named `GEMINI_API_KEY`.

In [ ]:
!pip install -q -U google-genai

import os, re, json, time, glob
import pandas as pd, joblib
from google import genai
from google.genai import types
from google.colab import userdata

NEEDED = ["indian_test_set.csv", "final_tfidf.joblib", "final_lr.joblib", "chakshu.txt",
          "npci_fraud_awareness.txt", "rbi_safe_digital_banking.txt", "kerala_police_1930.txt"]
for f in NEEDED:
    assert os.path.exists(f), f"Upload {f} first"

## 1. Knowledge-base chunks
Each source file is split into chunks of up to about 700 characters. The source name, URL and retrieval date travel with every chunk.

In [ ]:
MAX_CHARS = 700
HEADINGS = {"do's", "don'ts", "do's & don'ts for users"}

def norm(s):
    return s.replace("\u2019", "'").strip().lower()

def load_source(path):
    lines = open(path, encoding="utf-8").read().strip().splitlines()
    meta, body_start = {}, 0
    for i, line in enumerate(lines):
        m = re.match(r"\s*(Source|URL|Retrieved):\s*(.*)", line)
        if m:
            meta[m.group(1)] = m.group(2).strip()
        if len(meta) == 3:
            body_start = i + 1
            break
    assert len(meta) == 3, f"{path}: header is missing Source, URL or Retrieved"
    return meta, "\n".join(lines[body_start:]).strip()

def make_chunks(body):
    paras = [p.strip() for p in re.split(r"\n\s*\n", body) if p.strip()]
    chunks, cur = [], ""
    for p in paras:
        force_new = norm(p) in HEADINGS
        if cur and (force_new or len(cur) + len(p) + 2 > MAX_CHARS):
            chunks.append(cur)
            cur = ""
        cur = (cur + "\n\n" + p) if cur else p
    if cur:
        chunks.append(cur)
    merged = []
    for c in chunks:
        if merged and len(merged[-1]) < 60:
            merged[-1] += "\n\n" + c
        else:
            merged.append(c)
    return merged

chunks = []
for path in sorted(glob.glob("*.txt")):
    meta, body = load_source(path)
    for n, text in enumerate(make_chunks(body), start=1):
        chunks.append({"id": f"{path[:-4]}-{n}", "source": meta["Source"],
                       "url": meta["URL"], "retrieved": meta["Retrieved"], "text": text})
by_id = {c["id"]: c for c in chunks}
print("Chunks:", len(chunks))

Chunks: 10


## 2. Classifier and audit sample
The verdict that is passed to Gemini comes from the deployed TF-IDF + Logistic Regression model (notebook 04). The 20 audited messages are 10 scams and 10 legitimate messages, drawn with fixed seeds and shuffled so that every batch of 10 mixes both kinds.

In [ ]:
def clean(t):
    t = t.lower()
    t = re.sub(r"https?://\S+|www\.\S+", " urltoken ", t)
    t = re.sub(r"\b\d{10,}\b", " phonetoken ", t)
    t = re.sub(r"[\u20b9$\u00a3]\s?\d[\d,\.]*|\brs\.?\s?\d[\d,\.]*", " moneytoken ", t)
    t = re.sub(r"\d+", " numtoken ", t)
    return re.sub(r"\s+", " ", t).strip()

vec, clf = joblib.load("final_tfidf.joblib"), joblib.load("final_lr.joblib")

def classify_prob(message):
    return float(clf.predict_proba(vec.transform([clean(message)]))[0][1])

indian = pd.read_csv("indian_test_set.csv")
sample = (indian.groupby("label").sample(10, random_state=7)
                .sample(frac=1, random_state=1).reset_index(drop=True))
print("Audit sample:", len(sample), "messages |", sample["label"].value_counts().to_dict())

Audit sample: 20 messages | {0: 10, 1: 10}


## 3. Prompt, Gemini call and the resumable audit loop
The loop saves every result to `audit_cache.json` as it goes, stops at once on a daily-quota error, and can be run again to continue where it stopped.

In [ ]:
SYSTEM = """You write short scam warnings for a student awareness tool.
Rules:
1. Use ONLY the SOURCES provided for advice. Never use outside knowledge. Never invent phone numbers, websites or steps.
2. Every piece of advice must cite exactly one source_id from the SOURCES.
3. Give fewer advice items rather than unsupported ones. An empty advice list is allowed.
4. red_flags must be things actually visible in the message. Quote short phrases from it.
5. The verdict comes from a separate classifier and can be wrong. Never say a message is definitely a scam. Use wording like "looks like" or "may be".
6. Each advice item must be fully supported by its ONE cited source. If advice needs facts from two different sources, write two separate items, each with its own source_id.
7. Reply with JSON only, with no markdown fences, in exactly this shape:
{"red_flags": ["..."], "advice": [{"text": "...", "source_id": "..."}], "note": "..."}
Additional rule: If the verdict is 'likely legitimate', give at most ONE advice item and list red_flags only if they are clearly visible in the message. Do not alarm the user. If the verdict is 'unsure', say plainly that the tool cannot tell and suggest verifying through official channels."""

MODEL = "gemini-3.7-flash"      # the same model for the whole audit
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
SOURCES = "\n\n".join(f"[{c['id']}] {c['source']}\n{c['text']}" for c in chunks)

def explain_once(message, verdict, confidence):
    prompt = (f"SOURCES:\n{SOURCES}\n\nMESSAGE:\n{message}\n\n"
              f"CLASSIFIER VERDICT: {verdict} (confidence {confidence})")
    for attempt in range(3):
        try:
            r = client.models.generate_content(
                model=MODEL, contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM, response_mime_type="application/json", temperature=0.2))
            raw = re.sub(r"^```(?:json)?|```$", "", r.text.strip(), flags=re.M).strip()
            return json.loads(raw)
        except Exception as e:
            s = str(e)
            if "429" in s or "RESOURCE_EXHAUSTED" in s:
                raise RuntimeError("daily quota used up")
            if "503" in s or "UNAVAILABLE" in s:
                time.sleep(5 * (attempt + 1))
                continue
            raise
    raise RuntimeError("model busy")

CACHE = "audit_cache.json"

def run_audit(max_new=10):
    cache = json.load(open(CACHE, encoding="utf-8")) if os.path.exists(CACHE) else {}
    new = 0
    for n, r in sample.iterrows():
        key = str(n + 1)
        if key in cache or new >= max_new:
            continue
        p = classify_prob(r["text"])
        verdict = "likely scam" if p >= 0.65 else ("unsure" if p >= 0.35 else "likely legitimate")
        try:
            out = explain_once(r["text"], verdict, f"(scam probability {round(p, 2)})")
        except RuntimeError as e:
            print("Stopped at message", key, "-", e)
            break
        cache[key] = {"out": out, "verdict": verdict, "prob": round(p, 2), "model": MODEL}
        json.dump(cache, open(CACHE, "w", encoding="utf-8"), ensure_ascii=False)
        new += 1
        time.sleep(3)
    rows = []
    for key, v in cache.items():
        r = sample.iloc[int(key) - 1]
        flags = " | ".join(v["out"].get("red_flags", []))
        for a in (v["out"].get("advice", []) or [{"text": "", "source_id": ""}]):
            sid = a.get("source_id", "")
            rows.append({"msg_no": int(key), "true_label": r["label"], "message": r["text"],
                         "verdict": v["verdict"], "prob": v["prob"], "model": v["model"],
                         "red_flags": flags, "note": v["out"].get("note", ""),
                         "advice": a.get("text", ""), "source_id": sid,
                         "source_text": by_id[sid]["text"] if sid in by_id else ("SOURCE ID NOT FOUND" if sid else ""),
                         "advice_rating": "", "flags_rating": "", "comments": ""})
    pd.DataFrame(rows).to_csv("audit_sheet.csv", index=False, encoding="utf-8-sig")
    print("Total messages done:", len(cache), "of", len(sample), "| new this run:", new)

## 4. Running the audit (not needed to reproduce the report)
Uncomment the line below only if you accept that it uses up to 10 requests of the daily quota and will produce different wording from the rated sheet.

In [ ]:
# run_audit(max_new=10)

## 5. Summary recomputed from the rated sheet (the evidence for Section 8)
Upload `RedFlag_Audit_Sheet.xlsx` and run this cell. It reads the author's ratings, which are blue text on the `Audit` sheet.

In [ ]:
sheet = pd.read_excel("RedFlag_Audit_Sheet.xlsx", sheet_name="Audit")
tips = sheet["Advice rating"].fillna("").astype(str).str.strip()
n = len(tips)
print("Advice items rated:", n)
for code_, name in [("E", "exact"), ("P", "paraphrase"), ("S", "goes beyond the source"), ("U", "unsupported")]:
    c = int((tips == code_).sum())
    print(f"  {code_} ({name}): {c}  ({c / n:.0%})")
flags = sheet["Flags rating"].fillna("").astype(str).str.strip()
flags = flags[flags != ""]
print("Messages audited:", len(flags), "| red flags rated:", flags.value_counts().to_dict())
print("Invalid source ids:", int((sheet["Cited source text"].fillna("") == "SOURCE ID NOT FOUND").sum()))

Advice items rated: 36
  E (exact): 29  (81%)
  P (paraphrase): 6  (17%)
  S (goes beyond the source): 1  (3%)
  U (unsupported): 0  (0%)
Messages audited: 20 | red flags rated: {'G': 20}
Invalid source ids: 0
